# 1. Set Paths

In [1]:
import os
from pathlib import Path

In [2]:
# LORAKS 


# RESOURCES_PATH = os.path.join(DATASET_PATH, "example1", "resources")


DATASET_PATH = Path("/data/pt_02262/data/TH_bids")

SOURCE_PATH = DATASET_PATH / "source"
PREPARED_PATH = DATASET_PATH / "temp" / "LORAKS"
BIDSIFIED_PATH = DATASET_PATH / "bids" / "derivatives" / "LORAKS"
# RESOURCES_PATH = BIDSIFIED_PATH / "code" / "resources"
WORKING_DIR = Path.cwd()

In [3]:
print("Dataset path:", DATASET_PATH.is_dir())
print("Source path:", SOURCE_PATH.is_dir())
print("Prepared path:", PREPARED_PATH.is_dir())
print("Bidsified path:", BIDSIFIED_PATH.is_dir())
# print("Resources path:", RESOURCES_PATH.is_dir())
print("Working directory:", WORKING_DIR)

Dataset path: True
Source path: True
Prepared path: True
Bidsified path: True
Working directory: /data/u_kuegler_software/git/MPM_bidsification


# 2. Initialize `bidsme` and get the `logger` object
which which will control the logging of all bidsme functions:will control the logging of all bidsme functions:

In [4]:
import bidsme
logger = bidsme.init()

main(81) - INFO 
main(82) - INFO -------------- START bidsme ----------------
main(83) - INFO Mon May 12 14:11:45 2025
main(84) - INFO version: 1.9.0
bidsme.schema.BIDSschema(674) - INFO Loaded BIDS schema version 1.10.0


# 3. Prepare data set for bidsification

In [5]:
help(bidsme.prepare)

Help on function prepare in module bidsme.prepare:

prepare(
    source: str,
    destination: str,
    plugin_file: str = '',
    plugin_opt: dict = {},
    sub_list: list = [],
    sub_skip_tsv: bool = False,
    sub_skip_dir: bool = False,
    ses_skip_dir: bool = False,
    part_template: str = '',
    sub_prefix: str = '',
    ses_prefix: str = '',
    sub_no_dir: bool = False,
    ses_no_dir: bool = False,
    data_dirs: dict = {},
    dry_run: bool = False
) -> None
    Prepare data from surce folder and place it in
    sestination folder.

    Source folder is expected to have structure
    source/[subId/][sesId/][data/]file.
    Absence of subId and sesId levels must be communicated
    via sub_no_dir and ses_no_dir options. List of data
    folders must be given in data_dirs.

    Prepeared data will have structure
    destination/sub-<subId>/ses-<sesId>/<type>/<sequence>/file

    A list of treated subjects will be created/updated
    in destination/participants.tsv file

  

In [6]:
logger.setLevel("INFO")       
bidsme.prepare(str(SOURCE_PATH), str(PREPARED_PATH), 
               data_dirs={"nii_loraks_recon":"MRI",
                          # "nii_loraks/ernst_loraks":"MRI",
                          # "nii_loraks/pdw_loraks":"MRI",
                          # "nii_loraks/t1w_loraks":"MRI",
                          },
               plugin_file = str(WORKING_DIR / "plugins_bidsme" / "plugin_prepare_loraks_nk.py"),
               part_template = str(WORKING_DIR / "supplementary" / "table_templates" / "participants_nk.json"),
               plugin_opt = {"sessions_tsv_template": str(WORKING_DIR / "supplementary" / "table_templates" / "sessions_nk.json")}, 
               sub_list=["sub-005"] # only run on specified subjects (must be specified in BIDS notation)
              )                           
bidsme.tools.info.reporterrors(logger)
bidsme.tools.info.reseterrors(logger)

bidsme.prepare(192) - INFO -------------- Prepearing data -------------
bidsme.prepare(193) - INFO Source directory: /data/pt_02262/data/TH_bids/source
bidsme.prepare(194) - INFO Destination directory: /data/pt_02262/data/TH_bids/temp/LORAKS
bidsme.plugins.plugins(79) - INFO Loading module plugin_prepare_loraks_nk from /data/u_kuegler_software/git/MPM_bidsification/plugins_bidsme/plugin_prepare_loraks_nk.py
options passed to plugin:
- include_smaps: False
Loading sessions_nk.json from /data/u_kuegler_software/git/MPM_bidsification/supplementary/table_templates/sessions_nk.json.
          This functionality is not part of Bidsme, but implemented in a plugin. 
          It only works for processing all sessions of a subjects. Problems may 
          arise if the plugin is used for single sessions.
bidsme.bidsMeta.BidsTable(141) - INFO Created empty participants.tsv table
Subject ID derived from '/data/pt_02262/data/TH_bids/id_info/subject_ids.csv'.
Current subject: 37446.6e -> 002
bidsme

> **Note**: **apparently, it is not possible to specify a specific subset of sessions**
> + If the user wishes to rename subjects and/or sessions, it can be done with plug-in functions ```SubjectEP``` and ```SessionEP``` or by renaming directly folders in the prepared dataset.

# 4. Create the bidsmap.yaml

+ most tedious part of the process

In [ ]:
# help(bidsme.mapper)

In [7]:
PLUGIN_BIDS = WORKING_DIR / "plugins_bidsme" / "plugin_bidsify_loraks_nk.py"

In [8]:
bidsme.mapper(str(PREPARED_PATH), str(BIDSIFIED_PATH), plugin_file=str(PLUGIN_BIDS),
              plugin_opt={"bidsmap_step": True},
              sub_list=["sub-005"]
              # sub_skip_tsv=True,
              )
bidsme.tools.info.reporterrors(logger)
bidsme.tools.info.reseterrors(logger)

bidsme.mapper(253) - INFO ------------ Generating bidsmap ------------
bidsme.mapper(254) - INFO Current directory: /data/u_kuegler_software/git/MPM_bidsification
bidsme.mapper(255) - INFO Source directory: /data/pt_02262/data/TH_bids/temp/LORAKS
bidsme.mapper(256) - INFO Destination directory: /data/pt_02262/data/TH_bids/bids/derivatives/LORAKS
bidsme.mapper(272) - INFO loading template bidsmap bidsmap_template.yaml
bidsme.bidsmap._bidsmap(104) - WARNING Failed to find type EEG/BrainVision readed from /data/u_kuegler_software/miniforge3/envs/bidsme_env/lib/python3.13/site-packages/bidsme/heuristics/bidsmap_template.yaml. 
bidsme.mapper(291) - INFO loading working bidsmap /data/pt_02262/data/TH_bids/bids/derivatives/LORAKS/code/bidsme/bidsmap.yaml
bidsme.plugins.plugins(79) - INFO Loading module plugin_bidsify_loraks_nk from /data/u_kuegler_software/git/MPM_bidsification/plugins_bidsme/plugin_bidsify_loraks_nk.py
options passed to plugin:
- bidsmap_step: True, <class 'bool'>
- include_

+ open the created yaml file in VS Code
+ fix each warning/error, save the file, and repeat the code block above
    - find help at in the [Jupyter Notebooks in the bidsme tutorial](https://github.com/CyclotronResearchCentre/bidsme_tutorial) or in the docs [under Bidsmap creation](https://github.com/CyclotronResearchCentre/bidsme/blob/dev/doc/creating_map.md)
+ resume until there are no warnings left

> **Note:** to find a specific string in a file, use the command ```cat file.json | grep -i "string"``` **or** use ```less file.json``` and search using ```/string``` (```n``` will take you to the next entry & ```shift+n``` to the previous one; ```-I``` for case-insensitive)

> **Note:** Bidsme allow some limited transformation of data retrieved from header, these transformations are called actions and are defined in function ```action_value``` in file ```INSTALLATION_PATH/bidsme/Modules/common.py```.

```
Accepted actions:
    "": no action, return value
    int: cast value to int
    float: cast value to float
    str: cast value to string
    format<parameters>: apply python3 formatting
        mini-language to value, {:<parameters>}.format(value)
    scale<int>: apply a 10-based scale to value,
        value ** <int>
    mult<float>: multiply value
    div<float>: divide value
    round<int>: round value to given precision
```


+ The naming schema and sidecar json fields for a given modality (in this case MRI) are defined in $INSTALLATION_PATH/bidsme/Modules/MRI/_MRI.py. The list of entities is stored in modalities dictionary. If an image belongs for example to anat, bidsme will load the list of entities from modalities["anat"].

+ The optional model field will foce to use different list of entities from modalities dictionary. We will use the models extensively, while creating map for MPM part of the examle dataset.


# 5. Bidsification of the data set

In [9]:
MAP_FILE = str(BIDSIFIED_PATH / "code" / "bidsme" / "bidsmap.yaml")
PLUGIN_FILE_BIDS = str(WORKING_DIR / "plugins_bidsme" / "plugin_bidsify_loraks_nk.py")

-b = mapping file, --plugin = plugin

In [ ]:
!bidsme bidsify -help

In [10]:
# !bidsme bidsify $PREPARED_PATH $BIDSIFIED_PATH -b $MAP_FILE --plugin $PLUGIN_FILE_BIDS
!bidsme bidsify $PREPARED_PATH $BIDSIFIED_PATH -b $MAP_FILE --plugin $PLUGIN_FILE_BIDS --participants 'sub-005'
# !bidsme bidsify $PREPARED_PATH $BIDSIFIED_PATH -b $MAP_FILE --plugin $PLUGIN_FILE_BIDS --skip-existing

main(81) - INFO 
main(82) - INFO -------------- START bidsme ----------------
main(83) - INFO Mon May 12 14:25:46 2025
main(84) - INFO version: 1.9.0
bidsme.schema.BIDSschema(674) - INFO Loaded BIDS schema version 1.10.0
bidsme.bidsify(188) - INFO -------------- Prepearing data -------------
bidsme.bidsify(189) - INFO Source directory: /data/pt_02262/data/TH_bids/temp/LORAKS
bidsme.bidsify(190) - INFO Destination directory: /data/pt_02262/data/TH_bids/bids/derivatives/LORAKS
bidsme.bidsify(212) - WARNING Dataset description file 'dataset_description.json' not found in '/data/pt_02262/data/TH_bids/bids/derivatives/LORAKS'
bidsme.bidsify(218) - WARNING Dataset readme file 'README' not found in '/data/pt_02262/data/TH_bids/bids/derivatives/LORAKS'
bidsme.bidsify(233) - INFO loading bidsmap /data/pt_02262/data/TH_bids/bids/derivatives/LORAKS/code/bidsme/bidsmap.yaml
bidsme.plugins.plugins(79) - INFO Loading module plugin_bidsify_loraks_nk from /data/u_kuegler_software/git/MPM_bidsification

In [ ]:
help(bidsme.bidsify)

## Hints


The `004-al_mtflash3d_PDw` and `005-al_mtflash3d_PDw` are
the anatomical images
(suffix -- `MPM`)
taken using several echo times (`echo-1` ... `echo-6`),
and splitted into magnitude and phase components (`part-mag` and `part-phase`).
Additionaly, as for the PD-weighted images, the MT pulse wasn't used, we will
add the `mt-off` entity.
We will also add `flip-1` to the name, to mark that PDw images uses different
flip angle from T
So the final name will become:
`anat/sub-001_ses-s01530_acq-PDw_echo-1_mt-off_part-mag_MPM.nii`.
`anat/sub-001_ses-s01530_acq-PDw_echo-1_flip-1_mt-off_part-mag_MPM.nii`

The `002-al_mtflash3d_sensArray` and `003-al_mtflash3d_sensBody`
are [B1 fieldmaps](https://bids-specification.readthedocs.io/en/stable/99-appendices/11-qmri.html#rb1cor-specific-notes)
(suffix -- `RB1COR`),
taken for the PD-weighted images, using head and body coils
(`acq-headPDw` and `acq-bodyPDw`).
So their names will be simply: `fmap/sub-001_ses-s01530_acq-headPDw_RB1COR.nii`.

The similar names can be applied to T1w and MTw images and corresponding fieldmaps,
using corresponding `acq-` entities `acq-T1w` and `acq-MTw`.
For MTw images we alse need to use the `mt-on` entity.

Finally, `014-al_B1mapping` is the
[global B1 map](https://bids-specification.readthedocs.io/en/stable/99-appendices/11-qmri.html#tb1epi-specific-notes)
(suffix -- `TB1EPI`)
sets of images, taken with two echo times (`echo-1`, `echo-2`)
and several flip angles (`flip-01`, ... `flip-08`).
So the final name will become: `fmap/sub-001_ses-s01530_echo-1_flip-01_TB1EPI`